In [7]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer    
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

from xgboost import XGBClassifier   

import matplotlib.pyplot as plt 

In [4]:
def apply_mappings(df): 
    df = df.copy()
    grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7} 
    df['loan_grade'] = df['loan_grade'].map(grade_map)
    df['cb_person_default_on_file'] = df['cb_person_default_on_file'].map({'Y': 1, 'N': 0})
    return df


In [5]:
df = pd.read_csv('credit_risk_dataset.csv')
X = df.drop('loan_status', axis=1)
y = df['loan_status']

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [5]:
categorical_cols = ['person_home_ownership', 'loan_intent']
numeric_cols = [col for col in X_train.columns if col not in categorical_cols]

In [6]:
preprocessor = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), numeric_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))   
    ]), categorical_cols),
])

In [7]:
logistic_pipeline = Pipeline([
    ('mapper', FunctionTransformer(apply_mappings)),
    ('preprocessor', preprocessor),
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42))
])

logistic_pipeline.fit(X_train, y_train)
y_pred_proba_lr = logistic_pipeline.predict_proba(X_test)[:, 1]
roc_auc_lr  = roc_auc_score(y_test, y_pred_proba_lr)

In [8]:
xgb_pipeline = Pipeline([
    ('mapper', FunctionTransformer(apply_mappings)),
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(learning_rate = 0.1, max_depth = 4, n_estimators = 300, eval_metric='logloss', random_state=42))
])

xgb_pipeline.fit(X_train, y_train)
y_pred_proba_xgb = xgb_pipeline.predict_proba(X_test)[:, 1]
roc_auc_xgb  = roc_auc_score(y_test, y_pred_proba_xgb)

In [9]:
print(f'Logistic Regression ROC AUC: {roc_auc_lr:.2f}')
print(f'XGBoost ROC AUC: {roc_auc_xgb:.2f}')    

Logistic Regression ROC AUC: 0.87
XGBoost ROC AUC: 0.95


In [10]:
results = pd.DataFrame(y_pred_proba_xgb, columns=['pd'])
print(f'Portfolio Default Probability (PD): {results["pd"].mean()*100:.2f}%')

results['ead'] = X_test['loan_amnt'].values
print(f'Average EAD: ${results["ead"].mean():.2f}')

lgd_map = {'A': 0.25, 'B': 0.35, 'C': 0.45, 'D': 0.55, 'E': 0.65, 'F': 0.75, 'G': 0.85}
results['loan_grade'] = X_test['loan_grade'].values
results['lgd'] = results['loan_grade'].map(lgd_map)

results['el'] = results['pd'] * results['ead'] * results['lgd']
print(f'Total Expected Loss: ${results["el"].sum():,.2f}')
print(f'Average Expected Loss: ${results["el"].mean():.2f}')

loss_contribution = (results.groupby('loan_grade')['el'].sum() / results['el'].sum()).sort_values(ascending=False)
print('\nLoss Contribution by Loan Grade (%):')
print((loss_contribution * 100).round(2))

Portfolio Default Probability (PD): 21.68%
Average EAD: $9670.39
Total Expected Loss: $7,351,864.07
Average Expected Loss: $1128.11

Loss Contribution by Loan Grade (%):
loan_grade
D    37.93
B    17.77
C    15.57
E    13.08
A     6.22
F     5.06
G     4.38
Name: el, dtype: float64


In [11]:
results['pd_stress'] = results['pd'] * 1.5  
results['lgd_stress'] = results['lgd'] + 0.1        
results['el_stress'] = results['pd_stress'] * results['ead'] * results['lgd_stress']    

baseline_el = results['el'].sum()
stress_el = results['el_stress'].sum()

print(f"Baseline EL: ${baseline_el:,.2f}")
print(f"Stressed EL: ${stress_el:,.2f}")
print(f"Increase (%): {((stress_el / baseline_el - 1) * 100):.2f}%")

stress_loss_contribution = (results.groupby('loan_grade')['el_stress'].sum() / results['el_stress'].sum()).sort_values(ascending=False)
print('\nStressed Loss Contribution by Loan Grade (%):')
print((stress_loss_contribution * 100).round(2))

Baseline EL: $7,351,864.07
Stressed EL: $13,357,030.97
Increase (%): 81.68%

Stressed Loss Contribution by Loan Grade (%):
loan_grade
D    37.01
B    18.86
C    15.71
E    12.46
A     7.19
F     4.74
G     4.04
Name: el_stress, dtype: float64
